In [1]:
import sys
from pathlib import Path
root = Path.cwd().parents[1]
sys.path.append(str(root))

In [ ]:
from pprint import pprint
import os
import logging

debug = False
logger = logging.getLogger()
logging.basicConfig(
    level=logging.DEBUG if debug else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

from src.preprocessing import (
    Reader, HTTPReader,
    Embedder, HTTPEmbedder,
    FixedCharChunker, ParagraphChunker,
    Document, Page
)
from src.indexing import DataStore, ChunkStore, FlatVectorStore, Index, OpenSearchBM25Store
from src.utils import AutoEDAIndex
from src.llm import OpenAILLM

import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

from opensearchpy import OpenSearch
os_client = OpenSearch(hosts=[{"host": "localhost", "port": 9201}])

base_url = os.getenv('BASE_OPEN_AI_URL')
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Инициализация индекса

In [15]:
# Установка opensearch 
# docker pull opensearchproject/opensearch:latest
# docker run -d --name opensearch-local -p 9201:9200 -p 9601:9600 -e "discovery.type=single-node" -e "DISABLE_SECURITY_PLUGIN=true" -v opensearch-data:/usr/share/opensearch/data opensearchproject/opensearch:latest

# Проверка
! curl http://localhost:9201

{
  "name" : "7369a1c58d8f",
  "cluster_name" : "docker-cluster",
  "cluster_uuid" : "LZpLuUYoRaGm-RPDa_Kq3w",
  "version" : {
    "distribution" : "opensearch",
    "number" : "3.6.0",
    "build_type" : "tar",
    "build_hash" : "4ca747d8d47f80162db323019357447126732e35",
    "build_date" : "2026-04-04T06:02:53.715432718Z",
    "build_snapshot" : false,
    "lucene_version" : "10.4.0",
    "minimum_wire_compatibility_version" : "2.19.0",
    "minimum_index_compatibility_version" : "2.0.0"
  },
  "tagline" : "The OpenSearch Project: https://opensearch.org/"
}


In [4]:
# embedder = HTTPEmbedder()
# embedder.get_memory_usage()

In [5]:
# reader = HTTPReader()
# res = reader.read('/Users/danilaandreev/Documents/HSE/degree/data/raw_data/4293720391.pdf')

In [6]:
DATA_STORE_DIR = '../../data/index/documents'
CHUNK_STORE_DIR = '../../data/index/chunks'
VECTOR_STORE_DIR = '../../data/index/vectors'
SQLITE_DIR = '../../data/index/db'

# embedder = Embedder(
#     device='mps', 
#     model_name='ai-forever/FRIDA',
#     type_handlers={
#         'query': lambda text: f'search_query: {text}',
#         'document': lambda text: f'search_document: {text}',
#     }
# )
# reader=Reader(logger)

embedder = HTTPEmbedder(
    # logger=logger
)
reader=HTTPReader()
chunker=ParagraphChunker(
    logger=logger, 
    max_length=490, 
    overlap_sentences=1
)

dimensions = embedder.get_embeddings(['test']).shape[-1]


bm25store = OpenSearchBM25Store(
    client=os_client,
    index_name=f"test_index",
    logger=logger,
)

index = Index(
    datastore=DataStore(DATA_STORE_DIR, logger),
    vectorstore=FlatVectorStore(VECTOR_STORE_DIR, dimensions, logger),
    chunkstore=ChunkStore(CHUNK_STORE_DIR, logger),
    chunker=chunker,
    embedder=embedder,
    reader=reader,
    sqlite_path=SQLITE_DIR,
    bm25store=bm25store,
    logger=logger
)

2026-04-14 13:34:52,382 | INFO | opensearch | HEAD http://localhost:9201/test_index [status:200 request:0.007s]
2026-04-14 13:34:52,528 | INFO | root | Index vectorstore loaded


In [7]:
# index.sync_bm25_from_chunkstore()

In [8]:
# index.clear()
index.info()

2026-04-14 13:34:56,709 | INFO | opensearch | HEAD http://localhost:9201/test_index [status:200 request:0.004s]
2026-04-14 13:34:56,718 | INFO | opensearch | POST http://localhost:9201/test_index/_count [status:200 request:0.009s]
2026-04-14 13:34:56,719 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 75206}, 'vectorstore': {'vectors': 75768, 'dimensions': 1536}, 'index': {'points': 75206, 'points_without_chunk': 0, 'chunks_without_point': 0}, 'bm25store': {'backend': 'opensearch', 'index_name': 'test_index', 'exists': True, 'documents': 75206}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 75206},
 'vectorstore': {'vectors': 75768, 'dimensions': 1536},
 'index': {'points': 75206,
  'points_without_chunk': 0,
  'chunks_without_point': 0},
 'bm25store': {'backend': 'opensearch',
  'index_name': 'test_index',
  'exists': True,
  'documents': 75206}}

## Загрузка документов

In [ ]:
# SOURCE_DATA_DIR = '../../data/raw_data'
# sources = [os.path.join(SOURCE_DATA_DIR, name) for name in os.listdir(SOURCE_DATA_DIR)]

SOURCE_DATA_DIR = Path("../../data/raw_data").resolve()
sources = [str((SOURCE_DATA_DIR / name).resolve()) for name in os.listdir(SOURCE_DATA_DIR)]

print(f'Количество источников: {len(sources)}')

doc_ids = index.add_sources(sources)
index.save_vectorstore()

2026-04-11 18:38:27,389 | INFO | root | add_sources: 175 new sources, 0 skipped


Количество источников: 175


Indexing 4293731372.pdf | RSS 1499 MB | MPS 0/0 MB:   0%|          | 0/175 [00:00<?, ?it/s]

Indexing 4293747636.pdf | RSS 527 MB | MPS 0/0 MB:   5%|▌         | 9/175 [02:54<43:06, 15.58s/it]   2026-04-11 18:41:21,768 | INFO | root | File b36cbe061a750adc384e7d6cc916fe69f4ac365bc39c919753e0360f84faaf5b.json already exists
2026-04-11 18:41:21,769 | INFO | root | File 8314605f2d50e08b779916a34145f95d38863a647af91854b47247a245869c93.json already exists
2026-04-11 18:41:21,769 | INFO | root | File cddb5ab4c959b5f79597e8d46f22fecd40582dc66267a2d717a7896d0e62c1e8.json already exists
Indexing 4293749800.pdf | RSS 527 MB | MPS 0/0 MB:   6%|▌         | 10/175 [03:03<37:50, 13.76s/it]2026-04-11 18:41:31,441 | INFO | root | File 78317af53e3afc65514ab2ab6bd9757a2f10928722e3055574fe4c04bf0fe1c1.json already exists
2026-04-11 18:41:31,441 | INFO | root | File ae4310ffc8d1290b95db2630cd34105c2ad366774c3352e42c1c623b67a61c25.json already exists
Indexing 4293801871.pdf | RSS 531 MB | MPS 0/0 MB:   7%|▋         | 12/175 [03:26<35:04, 12.91s/it]2026-04-11 18:41:54,856 | INFO | root | File 141cbf

In [8]:
index.info()

2026-04-11 19:38:23,888 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 75206}, 'vectorstore': {'vectors': 75768, 'dimensions': 1536}, 'index': {'points': 75206, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 75206},
 'vectorstore': {'vectors': 75768, 'dimensions': 1536},
 'index': {'points': 75206,
  'points_without_chunk': 0,
  'chunks_without_point': 0}}

In [11]:
test_query = 'Нагрузка перекрытий'
res = index.search(test_query)

for r in res:
    print(r.chunk['text'])
    print()

2026-04-11 18:23:51,518 | INFO | root | HTTPEmbedder: batch 1/1 before request | memory={'cpu_rss_mb': 4203.359375, 'device': 'mps:0', 'mps_current_allocated_mb': 3141.0263671875, 'mps_driver_allocated_mb': 3700.53125}
2026-04-11 18:23:51,822 | INFO | root | HTTPEmbedder: batch 1/1 after request | memory={'cpu_rss_mb': 4209.96875, 'device': 'mps:0', 'mps_current_allocated_mb': 3141.0263671875, 'mps_driver_allocated_mb': 3702.53125}


8.3 Сосредоточенные нагрузки и нагрузки на перила 8.3.1 Несущие элементы перекрытий, покрытий, лестниц и балконов (лоджий) должны быть про­ верены на сосредоточенную вертикальную нагрузку, приложенную к элементу, в неблагоприятном по­ ложении на квадратной площадке со сторонами не более 10 см. Если в задании на проектирование на основании технологических решений не предусмотрены более высокие нормативные значения сосредоточенных нагрузок, их следует принимать: а) для перекрытий и лестниц — 1,5 кН; б) для чердачных перекрытий, покрытий, террас и балконов — 1,0 кН; в) для покрытий, по которым можно передвигаться только с помощью трапов и мостиков, — 0,5 кН.

8 Нагрузки от оборудования, людей, животных, складируемых материалов и изделий, транспортных средств Правила настоящего раздела распространяются на нагрузки от людей, животных, оборудования, изделий, материалов, временных перегородок, транспортных средств, действующие на перекрытия, покрытия, лестницы зданий и сооружений и полы на гр

# Baseline

In [9]:
eda = AutoEDAIndex(index)
df = eda.run()
# df

DOCUMENTS
- count: 175
- text_length_mean: 140687.2857142857
- text_length_median: 108903.0
- text_length_min: 1874
- text_length_max: 693714

CHUNKS
- count: 75206
- text_length_mean: 383.9260564316677
- text_length_median: 408.0
- text_length_min: 50
- text_length_max: 490
- text_length_std: 91.35188546875249
- words_mean: 52.615176980560065
- sentences_mean: 4.475480679733
- paragraphs_mean: 1.0

EMBEDDINGS
- matrix_rows: 75768
- matrix_cols: 1536
- dtype: float64
- dimension_std_mean: 0.022084718022281846
- dimension_std_min: 0.0
- dimension_std_max: 0.0291736261137144
- pairwise_similarity_pairs: 2870357028
- pairwise_similarity_min: -0.20334342320643395
- pairwise_similarity_max: 1.0000001773297225
- pairwise_similarity_mean: 0.2300295813031197
- pairwise_similarity_std: 0.08779688270887404



In [ ]:
from pydantic import BaseModel, Field
from typing import Optional, List
    
llm = OpenAILLM(client, model_name='gpt-5.4')

class Answer(BaseModel):
    answer: str
res = llm.parse('Скажи привет', Answer)
print(res)

2026-04-15 22:52:39,653 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


answer='Привет!'


In [10]:
query = 'Какие бывают нагрузки на перекрытия в многоэтажных зданиях?'
extracted = index.search(query, top_k=10)
# context = '\n\n'.join([f'{i+1}. Doc {e.chunk['doc_id']}: page {e.chunk['page_number']}\n{e.chunk['text']}' for i, e in enumerate(extracted)])
context = '\n\n'.join([f'{i+1}. {e.chunk['text']}' for i, e in enumerate(extracted)])
prompt = f'Ты отвечаешь на вопросы пользователя по строительной документации. Отвечай только на основе доступного контекста.\n\nЗапрос: {query}\n\nКонтекст: {context}'
class Answer(BaseModel):
    reasoning: List[str] = Field(..., description='3-5 предложений по обдумыванию контекста')
    answer: str = Field(..., description='Финальный ответ')
answer = llm.parse(prompt, Answer)
pprint(answer.answer)

2026-04-03 10:14:31,885 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


('На перекрытия в многоэтажных зданиях по приведенному контексту действуют:\n'
 '\n'
 '- постоянные нагрузки — от собственного веса несущих и ограждающих '
 'конструкций;\n'
 '- временные равномерно распределенные нагрузки на перекрытия;\n'
 '- временные сосредоточенные нагрузки на перекрытия;\n'
 '- нагрузки от людей, оборудования, изделий, материалов, временных '
 'перегородок, транспортных средств;\n'
 '- в составе общих расчетов здания также учитываются снеговые и ветровые '
 'нагрузки.\n'
 '\n'
 'Для перекрытий и лестниц в контексте отдельно указана сосредоточенная '
 'нагрузка 1,5 кН, если в задании на проектирование не предусмотрены более '
 'высокие значения.')


In [11]:
print(context)

1. редств Правила настоящего раздела распространяются на нагрузки от людей, животных, оборудования, изделий, материалов, временных перегородок, транспортных средств, действующие на перекрытия, покрытия, лестницы зданий и сооружений и полы на грунтах. Варианты загружения перекрытий этими нагрузками следует принимать в соответствии с предусмотренными условиями возведения и эксплуатации зданий, в наиболее неблагоприятном расчетном положении. Если на стадии проектирования данные об этих условиях недост

2. таны на восприятие постоянных нагрузок: - от собственного веса несущих и ограждающих конструкций; - временных равномерно распределенных и сосредоточенных нагрузок на перекрытия; - снеговых и ветровых нагрузок для данного района строительства. Нормативные значения перечисленных нагрузок, учитываемые неблагоприятные сочетания на­ грузок или соответствующих им усилий, предельные значения прогибов и перемещений конструкций, значения коэффициентов надежности по нагрузкам должны быть приняты в

In [11]:
# гибридный поиск
query = 'Какие бывают нагрузки на перекрытия в многоэтажных зданиях?'
extracted = index.search_hybrid(
    query,
    top_k_dense=5,
    top_k_bm25=5,
)
# context = '\n\n'.join([f'{i+1}. Doc {e.chunk['doc_id']}: page {e.chunk['page_number']}\n{e.chunk['text']}' for i, e in enumerate(extracted)])
context = '\n\n'.join([f'{i+1}. {e.chunk['text']}' for i, e in enumerate(extracted)])
prompt = f'Ты отвечаешь на вопросы пользователя по строительной документации. Отвечай только на основе доступного контекста.\n\nЗапрос: {query}\n\nКонтекст: {context}'
class Answer(BaseModel):
    reasoning: List[str] = Field(..., description='3-5 предложений по обдумыванию контекста')
    answer: str = Field(..., description='Финальный ответ')
answer = llm.parse(prompt, Answer)
pprint(answer.answer)

2026-04-14 13:38:25,477 | INFO | opensearch | HEAD http://localhost:9201/test_index [status:200 request:0.005s]
2026-04-14 13:38:25,515 | INFO | opensearch | POST http://localhost:9201/test_index/_search [status:200 request:0.037s]
2026-04-14 13:38:33,053 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


('На перекрытия в многоэтажных зданиях, согласно приведенному контексту, '
 'учитывают следующие нагрузки:\n'
 '\n'
 '- постоянные нагрузки — от собственного веса несущих и ограждающих '
 'конструкций;\n'
 '- временные равномерно распределенные нагрузки на перекрытия;\n'
 '- временные сосредоточенные нагрузки на перекрытия;\n'
 '- нагрузки от оборудования, мебели, людей, складируемых материалов и '
 'изделий;\n'
 '- в необходимых случаях — нагрузки от животных и транспортных средств;\n'
 '- для здания в целом также учитываются снеговые и ветровые нагрузки для '
 'района строительства.\n'
 '\n'
 'По контексту отдельно указано, что несущие элементы перекрытий должны '
 'проверяться на сосредоточенную вертикальную нагрузку, приложенную в '
 'неблагоприятном положении.\n'
 '\n'
 'Если нужны, могу также отдельно перечислить нормативные значения '
 'сосредоточенных нагрузок для перекрытий, лестниц, балконов и покрытий из '
 'приведенного контекста.')


In [12]:
print(context)

1. При необходимости учета влияния длительности нагрузок, при проверке на выносливость, уста­ лостной прочности и в других случаях, оговоренных в нормах проектирования конструкций и основа­ ний, устанавливаются пониженные нормативные значения нагрузок от оборудования, людей, животных и транспортных средств на перекрытия жилых, общественных и сельскохозяйственных зданий, от мо­ стовых и подвесных кранов, снеговых, температурных климатических воздействий.

2. 6.2 Конструкции и основания многоквартирного здания должны быть рассчитаны на восприятие постоянных нагрузок: - от собственного веса несущих и ограждающих конструкций; - временных равномерно распределенных и сосредоточенных нагрузок на перекрытия; - снеговых и ветровых нагрузок для данного района строительства.

3. Если в задании на проектирование на основании технологических решений не предусмотрены более высокие нормативные значения сосредоточенных нагрузок, их следует принимать: а) для перекрытий и лестниц — 1,5 кН; б) для чердач

In [13]:
# гибридный поиск
query = 'Какие бывают нагрузки на перекрытия в многоэтажных зданиях?'
extracted = index.search_bm25(
    query,
    top_k=5,
)
# context = '\n\n'.join([f'{i+1}. Doc {e.chunk['doc_id']}: page {e.chunk['page_number']}\n{e.chunk['text']}' for i, e in enumerate(extracted)])
context = '\n\n'.join([f'{i+1}. {e.chunk['text']}' for i, e in enumerate(extracted)])
prompt = f'Ты отвечаешь на вопросы пользователя по строительной документации. Отвечай только на основе доступного контекста.\n\nЗапрос: {query}\n\nКонтекст: {context}'
class Answer(BaseModel):
    reasoning: List[str] = Field(..., description='3-5 предложений по обдумыванию контекста')
    answer: str = Field(..., description='Финальный ответ')
answer = llm.parse(prompt, Answer)
pprint(answer.answer)

2026-04-14 13:41:24,722 | INFO | opensearch | HEAD http://localhost:9201/test_index [status:200 request:0.004s]
2026-04-14 13:41:24,732 | INFO | opensearch | POST http://localhost:9201/test_index/_search [status:200 request:0.009s]
2026-04-14 13:41:30,576 | INFO | httpx | HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"


('В доступном контексте виды нагрузок на перекрытия в многоэтажных зданиях не '
 'перечислены. Упоминаются только:\n'
 '- «наклонные междуэтажные перекрытия» в многоэтажных зданиях стоянок '
 'автомобилей;\n'
 '- проход инженерных коммуникаций через перекрытия;\n'
 '- определение высоты этажа как расстояния до оси ригеля перекрытия.\n'
 '\n'
 'Сведений о классификации или перечне нагрузок на перекрытия в приведённом '
 'контексте нет.')


In [14]:
print(context)

1. h <6 hs = 15 hs >30 Податливое /7/150 /7/200 /7/300 Обозначения, принятые в таблице Д.4: h — высота многоэтажных зданий, равная расстоянию от верха фундамента до оси ригеля покрытия; hs — высота этажа в одноэтажных зданиях, равная расстоянию от верха фундамента до низа стропильных конструкций; в многоэтажных зданиях: для нижнего этажа — равная расстоянию от верха фундамента до оси ригеля перекрытия; для остальных этажей — равная расстоянию между осями смежных ригелей.

2. а также подпор воздуха отдельными система­ ми в объем общих лестничных клеток и лифтовых шахт в соответствии с нормативными документами по пожарной безопасности 5.1.27 В многоэтажных зданиях стоянок автомобилей для перемещения автомобилей следует предусматривать рампы, пандусы, наклонные междуэтажные перекрытия или специальные лифты (ме­ ханизированные устройства). 7

3. В стоянках автомобилей требования к системам вентиляции следует принимать по указанным документам как для складских зданий, относящихся по взрывоп

In [ ]:
# TODO
# Посмотреть LAIME
# Сделать наглядную схему бенчмарка для хорошего описания в ВКР
# Сделать схему с описанием баз данных в приложении и связкой между ними

# Интерфейс ассистента:
# добавить логи мышления
# добавить кликабельную ссылку на документы при ответе + возможно выделение текста https://github.com/ScienciaLAB/streamlit-pdf-viewer
